In [1]:
pip install openai pydub

  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
from dotenv import load_dotenv
from openai import OpenAI

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "audio" / "input"
CHUNKS_DIR = BASE_DIR / "audio" / "chunks"
TRANSCRIPTS_DIR = BASE_DIR / "transcripts"

for folder in (INPUT_DIR, CHUNKS_DIR, TRANSCRIPTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing from the environment")

client = OpenAI(api_key=api_key)
print("Client initialized; working folders are ready.")

Client initialized; working folders are ready.


In [4]:
%pip install audioop-lts

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
from pydub import AudioSegment

audio_path = Path("MS072.mp3")
audio = AudioSegment.from_file(audio_path)
print(f"Loaded: {len(audio) / 1000:.1f}s, "
      f"{audio.channels} channels, {audio.frame_rate} Hz")

Loaded: 133.0s, 2 channels, 44100 Hz


In [6]:
from io import BytesIO

sample = audio[:30_000]  # pydub uses milliseconds
sample_buffer = BytesIO()
sample.export(sample_buffer, format="wav")
sample_buffer.name = "MS072_first_30s.wav"
sample_buffer.seek(0)

transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=sample_buffer,
)

print(type(transcription))
print(transcription.text)

<class 'openai.types.audio.transcription.Transcription'>
Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.


In [8]:
from io import BytesIO

baseline_text = transcription.text
prompt = "Miss Davis, government, garden"

sample_buffer = BytesIO()
sample.export(sample_buffer, format="wav")
sample_buffer.name = "MS072_first_30s.wav"
sample_buffer.seek(0)

prompted_transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=sample_buffer,
    prompt=  "Oral history interview with Miss Davis. She recalls an evening when "
    "she was working in her garden and a man asked, 'Is this Miss Davis?'",
)
prompted_text = prompted_transcription.text

baseline_path = TRANSCRIPTS_DIR / "MS072_first_30s_unprompted.txt"
prompted_path = TRANSCRIPTS_DIR / "MS072_first_30s_prompted.txt"

baseline_path.write_text(baseline_text, encoding="utf-8")
prompted_path.write_text(prompted_text, encoding="utf-8")

print("UNPROMPTED:\n", baseline_text)
print("\nPROMPTED:\n", prompted_text)
print(f"\nSaved to:\n{baseline_path}\n{prompted_path}")

UNPROMPTED:
 Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.

PROMPTED:
 Now you just pretend you hadn't talked to me any before and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that. Do you understand? But I know they come one evening and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.

Saved to:
/Users/fra/Documents/VScode/Week 4/Thurs - Lab/transcripts/MS072_first_30s_unprompted.txt
/Users/fra/Documents/VScode/Week 4/Thurs - Lab/transcripts/MS072_first_30s_prompted.txt


### Commentary: Comparing the transcripts

Both versions convey the same account and identify Miss Davis and her garden. The prompted version removes the comma after ‘before’, changes ‘..., you understand?’ to ‘.... Do you understand?’, and removes the comma after ‘one evening’. The remaining wording is effectively the same, so the prompt changed punctuation and one phrase rather than correcting a name or key content term. Without a verified reference transcript, we cannot say which version is more accurate.

In [9]:
CHUNK_MINUTES = 1  # Use 10 for long meeting recordings
CHUNK_MS = CHUNK_MINUTES * 60 * 1000

chunk_records = []

for i, start_ms in enumerate(range(0, len(audio), CHUNK_MS), start=1):
    end_ms = min(start_ms + CHUNK_MS, len(audio))
    chunk_path = CHUNKS_DIR / f"{audio_path.stem}_{CHUNK_MINUTES}min_{i:03}.mp3"

    audio[start_ms:end_ms].export(
        str(chunk_path),
        format="mp3",
        bitrate="128k",
    )

    chunk_records.append({
        "path": chunk_path,
        "start_ms": start_ms,
        "end_ms": end_ms,
    })

for chunk in chunk_records:
    duration = (chunk["end_ms"] - chunk["start_ms"]) / 1000
    print(f"{chunk['path'].name}: {duration:.1f}s, "
          f"{chunk['path'].stat().st_size / 1024:.1f} KB")


MS072_1min_001.mp3: 60.0s, 938.4 KB
MS072_1min_002.mp3: 60.0s, 938.4 KB
MS072_1min_003.mp3: 13.0s, 204.9 KB


### Visualize the audio chunks on a timeline

Run this after creating `chunk_records`. Each colored block shows a chunk’s position in the original recording.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

recording_seconds = len(audio) / 1000
chunk_step_seconds = CHUNK_MINUTES * 60

fig, ax = plt.subplots(figsize=(12, 2.5))
colors = plt.cm.Set3(np.linspace(0, 1, len(chunk_records)))

for i, chunk in enumerate(chunk_records):
    start = chunk["start_ms"] / 1000
    end = chunk["end_ms"] / 1000
    ax.barh(
        0,
        end - start,
        left=start,
        height=0.5,
        color=colors[i],
        edgecolor="black",
    )
    ax.text((start + end) / 2, 0, str(i + 1), ha="center", va="center", weight="bold")

ax.set_xlim(0, recording_seconds)
ax.set_yticks([])
ax.set_title(f"Audio chunks: {audio_path.name}")
ax.set_xlabel("Position in original recording (mm:ss)")

ticks = list(np.arange(0, recording_seconds, chunk_step_seconds))
if not ticks or ticks[-1] < recording_seconds:
    ticks.append(recording_seconds)
ax.set_xticks(ticks)
ax.set_xticklabels([f"{int(t // 60):02d}:{int(t % 60):02d}" for t in ticks])
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

for i, chunk in enumerate(chunk_records, start=1):
    duration = (chunk["end_ms"] - chunk["start_ms"]) / 1000
    print(f"Chunk {i}: {duration:.1f}s — {chunk['path'].name}")


In [10]:
import json

combined_segments = []
chunk_texts = []

for i, chunk in enumerate(chunk_records, start=1):
    with chunk["path"].open("rb") as audio_file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )

    chunk_texts.append(result.text.strip())
    offset_seconds = chunk["start_ms"] / 1000

    for segment in result.segments:
        combined_segments.append({
            "start": round(offset_seconds + segment.start, 2),
            "end": round(offset_seconds + segment.end, 2),
            "text": segment.text.strip(),
        })

    print(f"Transcribed chunk {i}/{len(chunk_records)}")

combined_text = " ".join(chunk_texts)

json_path = TRANSCRIPTS_DIR / "MS072_chunked_transcript.json"
json_path.write_text(
    json.dumps(
        {"source": audio_path.name, "text": combined_text, "segments": combined_segments},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

timestamped_text = "\n".join(
    f"[{s['start']:.2f}s - {s['end']:.2f}s] {s['text']}"
    for s in combined_segments
)
text_path = TRANSCRIPTS_DIR / "MS072_chunked_transcript.txt"
text_path.write_text(timestamped_text, encoding="utf-8")

print(f"Saved {len(combined_segments)} timestamped segments.")
print(json_path)
print(text_path)

Transcribed chunk 1/3
Transcribed chunk 2/3
Transcribed chunk 3/3
Saved 23 timestamped segments.
/Users/fra/Documents/VScode/Week 4/Thurs - Lab/transcripts/MS072_chunked_transcript.json
/Users/fra/Documents/VScode/Week 4/Thurs - Lab/transcripts/MS072_chunked_transcript.txt


### Print chunks with original timestamps

After the chunk transcription cell runs, this prints each chunk’s time range and its transcript segments using offsets from the full recording.


## Step 8: Export timestamped transcripts

This saves a readable timestamped TXT, numbered SRT subtitle cues, and structured JSON from `combined_segments`. It does not call the transcription API again.


In [ ]:
import json
from pathlib import Path


def format_human_timestamp(seconds):
    total_ms = max(0, int(round(float(seconds) * 1000)))
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    whole_seconds, milliseconds = divmod(remainder, 1000)
    return f"{hours:02d}:{minutes:02d}:{whole_seconds:02d}.{milliseconds:03d}"


def format_srt_timestamp(seconds):
    total_ms = max(0, int(round(float(seconds) * 1000)))
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    whole_seconds, milliseconds = divmod(remainder, 1000)
    return f"{hours:02d}:{minutes:02d}:{whole_seconds:02d},{milliseconds:03d}"


export_segments = []
for segment in sorted(combined_segments, key=lambda item: item["start"]):
    text = " ".join(str(segment["text"]).split())
    if not text:
        continue
    start = max(0.0, float(segment["start"]))
    end = max(float(segment["end"]), start + 0.001)
    export_segments.append({
        "start": round(start, 3),
        "end": round(end, 3),
        "text": text,
    })

base_name = audio_path.stem
readable_txt = TRANSCRIPTS_DIR / f"{base_name}_transcript_with_timestamps.txt"
srt_file = TRANSCRIPTS_DIR / f"{base_name}_transcript.srt"
json_file = TRANSCRIPTS_DIR / f"{base_name}_transcript.json"

text_lines = [
    f"Timestamped transcript: {audio_path.name}",
    f"Segments: {len(export_segments)}",
    "",
]
for segment in export_segments:
    text_lines.append(
        f"[{format_human_timestamp(segment['start'])} --> "
        f"{format_human_timestamp(segment['end'])}] {segment['text']}"
    )
readable_txt.write_text("\n".join(text_lines) + "\n", encoding="utf-8")

srt_blocks = []
for cue_number, segment in enumerate(export_segments, start=1):
    start_ms = max(0, int(round(segment["start"] * 1000)))
    end_ms = max(int(round(segment["end"] * 1000)), start_ms + 1)
    start_time = format_srt_timestamp(start_ms / 1000)
    end_time = format_srt_timestamp(end_ms / 1000)
    srt_blocks.append(
        f"{cue_number}\n{start_time} --> {end_time}\n{segment['text']}"
    )
srt_file.write_text("\n\n".join(srt_blocks) + "\n", encoding="utf-8")

json_payload = {
    "source_file": audio_path.name,
    "model": "whisper-1",
    "text": combined_text,
    "segments": [
        {"id": i, **segment}
        for i, segment in enumerate(export_segments, start=1)
    ],
}
json_file.write_text(
    json.dumps(json_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# Read the files back to confirm that the saved exports are valid and complete.
json_check = json.loads(json_file.read_text(encoding="utf-8"))
srt_check = srt_file.read_text(encoding="utf-8")
txt_check = readable_txt.read_text(encoding="utf-8")
srt_cue_count = len([block for block in srt_check.strip().split("\n\n") if block.strip()])
assert len(json_check["segments"]) == len(export_segments)
assert srt_cue_count == len(export_segments)
assert txt_check.startswith(f"Timestamped transcript: {audio_path.name}")

print(f"TXT:  {readable_txt} ({readable_txt.stat().st_size} bytes)")
print(f"SRT:  {srt_file} ({srt_cue_count} cues)")
print(f"JSON: {json_file} ({len(json_check['segments'])} segments)")
print("\nSRT preview:\n")
print(srt_check.split("\n\n", 1)[0])


In [11]:
def format_timestamp(seconds):
    hours, remainder = divmod(float(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}"
    return f"{int(minutes):02d}:{seconds:05.2f}"

for i, chunk in enumerate(chunk_records, start=1):
    chunk_start = chunk["start_ms"] / 1000
    chunk_end = chunk["end_ms"] / 1000
    duration = chunk_end - chunk_start
    print(
        f"\nCHUNK {i}: {format_timestamp(chunk_start)} - "
        f"{format_timestamp(chunk_end)} ({duration:.1f}s)"
    )

    chunk_segments = [
        segment for segment in combined_segments
        if chunk_start <= segment["start"] < chunk_end
    ]
    for segment in chunk_segments:
        print(
            f"  [{format_timestamp(segment['start'])} - "
            f"{format_timestamp(segment['end'])}] {segment['text']}"
        )



CHUNK 1: 00:00.00 - 01:00.00 (60.0s)
  [00:00.00 - 00:06.00] Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here.
  [00:06.00 - 00:12.00] Well, now I don't remember the day or the month or nothing like that, you understand?
  [00:12.00 - 00:19.00] But I know they come one evening, and I just was fixing to quit working in my garden.
  [00:19.00 - 00:27.00] And he comes and he says, well, says, is this Miss Davis?
  [00:27.00 - 00:34.00] And I said, that's what they call me. Other words, Aunt Lou, I suggest.
  [00:34.00 - 00:46.00] He says, well, since I'm taking pictures around, says the government's going to take over everything, and James was going to have to move out.
  [00:46.00 - 00:50.00] I said, all this many years? I said, you talk crazy.
  [00:50.00 - 00:59.00] And he lay, he says, no, says, I'm here just to tell you, says, friendly talk.

CHUNK 2: 01:00.00 - 02:00.00 (60.0s)
  [01:00.00 - 01:08.00] And he says, then